# Prescriptive Analytics

Turning predictions into optimal decisions – building end-to-end predict-then-optimize pipelines in Operations Research.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import pulp

np.random.seed(42)

## Step 1: Demand Prediction

In [ ]:
# Generate synthetic features and demand data
n = 2000
data = pd.DataFrame({
    'price': np.random.uniform(20, 80, n),
    'promotion': np.random.binomial(1, 0.4, n),
    'holiday': np.random.binomial(1, 0.15, n),
    'competitor_price': np.random.uniform(25, 85, n),
    'day_of_week': np.random.randint(0, 7, n),
    'marketing_spend': np.random.uniform(800, 6000, n)
})

# Realistic demand function
data['demand'] = (
    250 
    - 3.2 * data['price'] 
    + 85 * data['promotion'] 
    + 110 * data['holiday'] 
    - 2.1 * (data['price'] - data['competitor_price']) 
    + 0.018 * data['marketing_spend']
    + np.random.normal(0, 35, n)
).clip(lower=30)

X = data.drop('demand', axis=1)
y = data['demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = RandomForestRegressor(n_estimators=400, random_state=42)
model.fit(X_train, y_train)

print(f"Prediction model trained. Test MAE: {(abs(model.predict(X_test) - y_test)).mean():.2f}")

## Step 2: Prescriptive Optimization (Newsvendor-style Allocation)

In [ ]:
# Future scenarios for next period
future_scenarios = pd.DataFrame({
    'price': [55, 62, 48],
    'promotion': [1, 0, 1],
    'holiday': [0, 1, 0],
    'competitor_price': [58, 60, 52],
    'day_of_week': [2, 5, 6],
    'marketing_spend': [4200, 1800, 5100]
})

# Predict demand for each scenario
predicted_demands = model.predict(future_scenarios)
print("Predicted demands:", np.round(predicted_demands, 1))

In [ ]:
# Optimization: Decide order quantities across scenarios (two-stage style)
prob = pulp.LpProblem("Prescriptive_Newsvendor", pulp.LpMinimize)

# First-stage decision: order quantity (here-and-now)
x = pulp.LpVariable("order_quantity", lowBound=0, cat='Continuous')

# Costs
c_o = 4.0   # overage cost per unit
c_u = 14.0  # underage cost per unit

# Expected recourse cost
expected_cost = 0
for d in predicted_demands:
    over = pulp.LpVariable(f"over_{d:.0f}", lowBound=0)
    under = pulp.LpVariable(f"under_{d:.0f}", lowBound=0)
    
    prob += over >= x - d
    prob += under >= d - x
    
    expected_cost += (c_o * over + c_u * under) / len(predicted_demands)

prob += 10 * x + expected_cost   # 10 = purchase cost per unit

prob.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"Optimal order quantity: {pulp.value(x):.1f}")
print(f"Total expected cost: {pulp.value(prob.objective):.2f}")

## End-to-End Pipeline Summary

In [ ]:
# Full pipeline function
def predict_then_optimize(new_features, c_o=4.0, c_u=14.0):
    pred_d = model.predict(new_features)[0]
    
    # Simple critical fractile for illustration
    critical = c_u / (c_o + c_u)
    order_qty = pred_d   # can be improved with safety stock or full optimization
    
    return pred_d, order_qty

# Test on one new scenario
new_case = pd.DataFrame({
    'price': [58],
    'promotion': [1],
    'holiday': [0],
    'competitor_price': [61],
    'day_of_week': [3],
    'marketing_spend': [3900]
})

pred, order = predict_then_optimize(new_case)
print(f"Predicted demand: {pred:.1f} | Recommended order: {order:.0f}")

## Exercises

- Replace the simple newsvendor with a full multi-product production planning model.
- Use stochastic programming (from Chapter 6) with predicted demand distributions.
- Add uncertainty quantification (e.g., prediction intervals) and robust optimization.
- Deploy the pipeline as a function that can be called from a web dashboard.

This concludes the core technical chapters. The next section will focus on **case studies** and real-world applications.